In [1]:
import pandas as pd

# Import raw data for preprocessing

In [2]:
filename = "Forza Horizon 5 Car List"
tables = pd.read_html("raw-website/" + filename + ".htm")
df = tables[0]
df.head(10)

,Car Model,Car Type,Collect,Added,Nickname,ID
0,2017 Abarth 124 Spider,Modern Sports Cars,"Backstage, Seasonal",Series 24,Abarth 124 '17,2740
1,2016 Abarth 695 Biposto,Hot Hatch,"Backstage, Seasonal",Series 24,Abarth 695 '16,2489
2,1980 Abarth Fiat 131,Classic Rally,"Backstage, Seasonal",Series 24,Abarth 131,1124
3,1968 Abarth 595 esseesse,Cult Cars,Autoshow,Series 24,Abarth 595 '68,2017
4,2017 Acura NSX,Modern Supercars,Autoshow,Launch,Acura NSX '17,2352
5,2002 Acura RSX Type-S,Retro Hot Hatch,Autoshow,Launch,Acura RSX,422
6,2001 Acura Integra Type-R,Retro Hot Hatch,Autoshow,Launch,Acura Integra,368
7,2017 Alfa Romeo Giulia Quadrifoglio,Super Saloons,Autoshow,Series 24,Alfa Giulia '17,2542
8,2014 Alfa Romeo 4C,Modern Sports Cars,"Backstage, Seasonal",Series 24,Alfa Romeo 4C,2038
9,2007 Alfa Romeo 8C Competizione,GT Cars,"Backstage, Seasonal",Series 24,Alfa Romeo 8C,1032


## Remove redundant columns

In [ ]:
columns_to_drop = [1, 2, 3, 4]
df = df.drop(df.columns[columns_to_drop], axis=1)
df.head()

## Expand abbreviated columns

In [ ]:
df = df.rename(columns={"Sp": "Speed", "Ha": "Handling", "Ac": "Accel", "La": "Launch", "Br": "Braking", "Of": "OffRd"})
df.head()

In [ ]:
df.info()

## Remove stat columns from df_lite

In [ ]:
columns_to_drop = [2, 3, 4, 5, 6, 7]
df_lite = df.drop(df.columns[columns_to_drop], axis=1)
df_lite.head()

## Split original Value column into Value and Rarity

In [ ]:
df_lite.iloc[df_lite["Value"].str.contains("?", regex=False)].head()

In [ ]:
df_lite[["Value", "Rarity"]] = df["Value"].str.split("CR", n=1, expand=True)
df_lite = df_lite.iloc[:, [0, 1, 3, 2]]

df_lite["Value"] = df_lite["Value"].str.replace(",", "")
df_lite["Value"] = df_lite["Value"].astype(int)
df_lite.head()

## Split original Vehicle column into Brand and Vehicle

### Isolate rows for 2-word vehicle brand names

In [ ]:
df1 = df_lite.iloc[df_lite["Vehicle"].str.contains("Alfa|Aston|Auto Union|Automobili Pininfarina|Extreme E|Formula Drift|Forsberg Racing|Funco Motorsports|Land Rover|Local Motors|RJ Anderson|SIERRA Cars|Spania GTA|Universal Studios|W Motors")]
df1.head()

#### Replace first whitespace with underscore

In [ ]:
df1["Vehicle"] = df1["Vehicle"].str.replace(" ", "_", n=1)
df1.head()

### Separate filtering for Hot Wheels

In [ ]:
df_hw = df_lite.iloc[df_lite["Vehicle"].str.contains("Hot Wheels")]
df_hw = df_hw.loc[~df_hw["Vehicle"].str.contains("Brabham|Chevrolet|Hennessey|Mosler|SUBARU|Schuppan|SIERRA")]
df_hw.head()

In [ ]:
df_hw["Vehicle"] = df_hw["Vehicle"].str.replace(" ", "_", n=1)
df_hw.head()

### Isolate rows for 3-word vehicle brand names

In [ ]:
df2 = df_lite.iloc[df_lite["Vehicle"].str.contains("AMG Transport|Casey Currie|Fast and Furious|Gordon Murray|Lynk & Co")]
df2.head()

#### Replace first two whitespace with underscore

In [ ]:
df2["Vehicle"] = df2["Vehicle"].str.replace(" ", "_", n=2)
df2.head()

### Update df_lite with altered rows

In [ ]:
df_lite.update(df1)
df_lite.head()

In [ ]:
df_lite.update(df2)
df_lite.head()

In [ ]:
df_lite.update(df_hw)
df_lite.head()

### Perform split

In [ ]:
df_lite[["Brand", "Vehicle"]] = df_lite["Vehicle"].str.split(" ", n=1, expand=True)
df_lite = df_lite.iloc[:, [4, 0, 1, 2, 3]]
df_lite.head()

#### Error checking

In [ ]:
df_verify = df_lite.iloc[df_lite["Vehicle"].str.contains("_")]
df_verify.head()

#### Replace underscores in Brand column with whitespace

In [ ]:
df_lite["Brand"] = df_lite["Brand"].str.replace("_", " ")
df_lite.head()

## Split into 3 columns using str.extract and regex

### Filter "Car Mastery" rows separately

In [ ]:
df_cm = df_lite.iloc[df_lite["Vehicle"].str.contains("Car Mastery")]
df_cm.head()

In [ ]:
df_cm[["Vehicle", "Year", "Collection"]] = df_cm["Vehicle"].str.extract(r'^(.*?)(\d{4})(.*)$')
df_cm = df_cm.iloc[:, [0, 1, 5, 2, 6, 3, 4]]
df_cm.head()

### Perform general splitting

In [ ]:
df_lite[["Vehicle", "Year", "Collection"]] = df_lite["Vehicle"].str.extract(r'^(.*)(\d{4})(.*)$')
df_lite = df_lite.iloc[:, [0, 1, 5, 2, 6, 3, 4]]
df_lite.head()

#### Update with df_cm

In [ ]:
df_lite.update(df_cm)
df_lite.head()

In [ ]:
df_lite.iloc[df_lite["Collection"].str.contains("Car Mastery")].head()

## Sort by Brand then Year and Vehicle (ascending)

In [ ]:
df_lite.dtypes

In [ ]:
df_lite = df_lite.sort_values(["Brand", "Year", "Vehicle"], ascending=[True, True, True], key=lambda col: col.str.lower())
df_lite = df_lite.reset_index(drop=True)
df_lite.head(10)

## Clean PI column

In [ ]:
df_lite[["PI1", "PI2"]] = df_lite["PI"].str.split(r'(?<=D)|(?<=C)|(?<=B)|(?<=A)|(?<=S1)|(?<=S2)', n=1,
                                                                  expand=True)
df_lite.head()

### Stats

In [ ]:
df_lite["PI1"].value_counts()
# "A" is the most common class
# followed by S1
# then B

In [ ]:
df_lite["PI2"].value_counts()
# A 800 is the most common PI value

In [ ]:
df_lite["Year"].value_counts()

In [ ]:
df_lite["Value"].value_counts()

In [ ]:
df_lite["Rarity"].value_counts()

In [ ]:
df_lite["PI"] = df_lite["PI1"] + " " + df_lite["PI2"]
df_lite = df_lite.drop(["PI1", "PI2"], axis="columns")
df_lite.head()

In [ ]:
df_lite["Owned"] = pd.Series(dtype="string")
df_lite.head()

### Convert dataframe to csv

In [ ]:
df_lite.to_csv("FH5/FH5-0-masterlist.csv", index=False, encoding="utf-8-sig")